# Baseline B: fine-tuned pretrained encoders (DiCo-NLI Track 1)

**Run on Colab with a GPU:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.

What this notebook does:
1. Clones the organizers' repo (it has the data and the official scorer).
2. Computes MiniLM sentence-embedding similarity for the EDA (`semantic_similarity.csv`).
3. **Fine-tunes** BERT-base and DeBERTa-v3-base as cross-encoders: input is `[CLS] text1 [SEP] text2 [SEP]`, output is one of 4 labels. All weights are updated (full fine-tuning, not frozen features).
4. Scores dev predictions with the official scorer (Weighted F1, SoftCons, HardCons).
5. Zips everything into `colab_results.zip` for download. Unzip it into the project's `results/` folder.

Protocol: fixed hyperparameters and fixed epoch count, no early stopping or tuning on dev (dev is our only held-out set, so we do not select on it).

In [ ]:
!pip -q install sentence-transformers sentencepiece protobuf
!git clone -q https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git
import sys, os, json, random, time
sys.path.insert(0, "SemEval-2027-Task-2-DiCo-NLI")
os.makedirs("results/predictions", exist_ok=True)
import numpy as np, pandas as pd, torch
from evaluation_functions import score_files
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Data (Track 1, official train/dev split, no re-splitting)

In [ ]:
D = "SemEval-2027-Task-2-DiCo-NLI/final_data"
REF = {s: f"{D}/{s}/dico_nli_{s}_track1_reference.csv" for s in ["train", "dev"]}
train = pd.read_csv(REF["train"], keep_default_na=False)
dev = pd.read_csv(REF["dev"], keep_default_na=False)
LABELS = ["EQUIVALENCE", "FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "NEGATIVE_OTHER"]
L2I = {l: i for i, l in enumerate(LABELS)}
print(len(train), len(dev)); train.label.value_counts()

## 2. Semantic similarity for the EDA (frozen sentence encoder, no training)

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
both = pd.concat([train, dev], ignore_index=True)
ph = list(pd.unique(pd.concat([both.text1, both.text2])))
E = dict(zip(ph, st.encode(ph, batch_size=256, normalize_embeddings=True)))
both["cos"] = [float(E[a] @ E[b]) for a, b in zip(both.text1, both.text2)]
both[["instance_id", "cos"]].to_csv("results/semantic_similarity.csv", index=False)
print(both.groupby("label").cos.mean().round(3))
del st; torch.cuda.empty_cache()

## 3. Fine-tuning loop (plain PyTorch, so every step is visible)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

CFG = dict(epochs=6, lr=3e-5, batch_size=16, max_len=64, warmup=0.1, weight_decay=0.01)
MODELS = ["bert-base-uncased", "microsoft/deberta-v3-base"]
SEEDS = [13, 42, 87]   # 3 seeds -> report mean ± std. Use [42] if short on time.

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def loader(df, tok, shuffle):
    enc = tok(list(df.text1), list(df.text2), truncation=True, max_length=CFG["max_len"])
    y = df.label.map(L2I).tolist()
    keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    items = [{**{k: enc[k][i] for k in keys}, "labels": y[i]} for i in range(len(df))]
    collate = lambda b: {**tok.pad([{k: x[k] for k in keys} for x in b], return_tensors="pt"),
                         "labels": torch.tensor([x["labels"] for x in b])}
    return DataLoader(items, batch_size=CFG["batch_size"], shuffle=shuffle, collate_fn=collate)

@torch.no_grad()
def predict(model, dl):
    model.eval(); probs = []
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        probs.append(torch.softmax(model(**b).logits.float(), -1).cpu())
    return torch.cat(probs).numpy()

def official(pred_labels, name):
    path = f"results/predictions/{name}.csv"
    pd.DataFrame({"instance_id": dev.instance_id, "label": pred_labels}).to_csv(path, index=False)
    r = score_files(REF["dev"], path).to_dict()
    return path, dict(weighted_f1=r["weighted_f1"], soft_cons=r["soft_cons"], hard_cons=r["hard_cons"],
                      macro_f1=r["classification"]["macro_f1"], accuracy=r["classification"]["accuracy"])

def run(model_name, seed):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS)).to(DEVICE)
    tr_dl, dv_dl = loader(train, tok, True), loader(dev, tok, False)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    total = len(tr_dl) * CFG["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * total), total)
    curve = []
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); losses = []
        for b in tr_dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = model(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
            losses.append(loss.item())
        probs = predict(model, dv_dl)
        short = model_name.split("/")[-1]
        _, s = official([LABELS[i] for i in probs.argmax(1)], f"_tmp")
        curve.append({"model": short, "seed": seed, "epoch": ep, "train_loss": float(np.mean(losses)), **s})
        print(f"{short} seed={seed} ep={ep} loss={np.mean(losses):.3f} " + " ".join(f"{k}={v:.3f}" for k, v in s.items()))
    # final model = last epoch (fixed in advance, no selection on dev)
    name = f"{short}_seed{seed}"
    path, s = official([LABELS[i] for i in probs.argmax(1)], name)
    pd.DataFrame(probs, columns=LABELS).assign(instance_id=dev.instance_id.values).to_csv(
        f"results/predictions/{name}_probs.csv", index=False)
    del model; torch.cuda.empty_cache()
    return {"model": short, "seed": seed, **s}, curve

## 4. Train and score

In [ ]:
finals, curves = [], []
for m in MODELS:
    for s in SEEDS:
        t = time.time(); f, c = run(m, s); finals.append(f); curves += c
        print(f"  done in {time.time()-t:.0f}s")
finals = pd.DataFrame(finals); curves = pd.DataFrame(curves)
finals.to_csv("results/transformer_results_per_seed.csv", index=False)
curves.to_csv("results/transformer_training_curves.csv", index=False)
summary = finals.groupby("model")[["weighted_f1", "soft_cons", "hard_cons", "macro_f1", "accuracy"]].agg(["mean", "std"]).round(3)
summary.to_csv("results/transformer_results_summary.csv")
summary

## 5. Download results (unzip into the project's `results/` folder)

In [ ]:
import os
if os.path.exists("results/predictions/_tmp.csv"): os.remove("results/predictions/_tmp.csv")
json.dump(CFG, open("results/transformer_config.json", "w"), indent=2)
!cd results && zip -qr ../colab_results.zip .
from google.colab import files
files.download("colab_results.zip")